# Challenge — Red neuronal con Keras 3
### Universidad EAFIT | SI3003 — Introducción a la Inteligencia Artificial

---

## Objetivo

En clase construimos una red neuronal de clasificación con **Keras 3**.

En este ejercicio vas a aplicar la misma receta sobre un dataset diferente:

> **Olivetti Faces**

El dataset contiene imágenes de rostros en escala de grises. El objetivo será identificar a cuál persona pertenece cada imagen.

No necesitas diseñar un pipeline de datos nuevo. La carga, división y visualización están resueltas.

Tu trabajo se concentra en completar las etapas principales de Keras:

```text
Datos
  ↓
Modelo
  ↓
Loss + Optimizer
  ↓
Entrenamiento
  ↓
Evaluación
  ↓
Predicción
```

---

## ¿Qué debes completar?

Las celdas marcadas con `# TODO` contienen espacios que debes completar.

El resto del código está preparado para que puedas concentrarte en los conceptos vistos en clase.


---
## 0. Importar librerías

Usaremos:

- **Keras 3** para construir y entrenar la red.
- **scikit-learn** para cargar Olivetti Faces y dividir los datos.
- **NumPy** para manipular arrays.
- **Matplotlib** para visualizar imágenes.


In [ ]:
import keras
from keras import layers

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split

SEED = 42
keras.utils.set_random_seed(SEED)

print(f"Keras version : {keras.__version__}")
print(f"Backend       : {keras.backend.backend()}")


---
# 1. Dataset: Olivetti Faces

Olivetti Faces contiene:

- **400 imágenes**
- imágenes de **64 × 64 píxeles**
- escala de grises
- **40 personas**
- **10 imágenes por persona**

La etiqueta indica la identidad de la persona:

```text
0, 1, 2, ..., 39
```

Cada imagen ya viene representada con valores flotantes aproximadamente en el rango:

$$
[0,1]
$$

Por eso, en este ejercicio **no necesitamos normalizar dividiendo por 255**.


In [ ]:
# Esta parte está resuelta.

faces = fetch_olivetti_faces(
    shuffle=True,
    random_state=SEED
)

X = faces.images
y = faces.target

print("X:", X.shape)
print("y:", y.shape)
print("Valor mínimo:", X.min())
print("Valor máximo:", X.max())
print("Número de clases:", len(np.unique(y)))


## 1.1 Dividir entrenamiento y prueba

Como solo tenemos 10 imágenes por persona, debemos asegurarnos de que todas las identidades aparezcan tanto en entrenamiento como en prueba.

Usaremos una división **estratificada**:

```text
80% entrenamiento
20% prueba
```

La opción:

```python
stratify=y
```

mantiene la proporción de cada clase en ambos conjuntos.

Esta parte está resuelta.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nPersonas en train:", len(np.unique(y_train)))
print("Personas en test :", len(np.unique(y_test)))


### Pregunta 1

Cada imagen tiene tamaño 64 × 64. Cuando apliquemos `Flatten()`, ¿cuántos valores tendrá cada ejemplo?

**Respuesta:** 64 × 64 = **4096 valores**.

---
## 1.2 Visualizar algunas imágenes

Esta parte está resuelta.

Observa que varias fotografías corresponden a la misma persona con pequeñas variaciones de expresión, iluminación y orientación.


In [ ]:
plt.figure(figsize=(12, 6))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(f"Persona {y_train[i]}")
    plt.xticks([])
    plt.yticks([])

plt.tight_layout()
plt.show()


---
# 2. Construir la red neuronal

Usaremos una arquitectura fully-connected similar a la vista en clase:

```text
Imagen 64×64
     ↓
Flatten
     ↓
4096 valores
     ↓
Dense(64)
     ↓
ReLU
     ↓
Dense(40)
     ↓
Logits
```

La última capa debe tener una salida por cada clase.

> **Importante:** no añadas `Softmax` al final. Trabajaremos directamente con **logits**.


In [ ]:
# TODO 1 — Completar la arquitectura.  --> RESUELTO
model = keras.Sequential([
    keras.Input(shape=(64, 64)),           # rostro 64x64 en escala de grises
    layers.Flatten(),                      # -> 4096 valores
    layers.Dense(64, activation="relu"),
    layers.Dense(40)                       # 40 personas (logits)
])

model.summary()

In [ ]:
# Validación TODO 1

assert isinstance(model, keras.Model)

# Dense(64): 4096*64 + 64
# Dense(40): 64*40 + 40
expected_params = (4096 * 64 + 64) + (64 * 40 + 40)

assert model.count_params() == expected_params, (
    f"Se esperaban {expected_params:,} parámetros, "
    f"pero el modelo tiene {model.count_params():,}."
)

print("✓ Arquitectura correcta.")
print(f"Parámetros entrenables: {model.count_params():,}")


### Pregunta 2
¿Por qué necesitamos `Flatten()` antes de la primera capa `Dense`?

**Respuesta:** la capa `Dense` requiere un vector 1D; `Flatten()` transforma la imagen 64×64 en un vector de 4096 valores para poder conectarla.

### Pregunta 3
¿Por qué la última capa tiene **40 neuronas**?

**Respuesta:** porque el dataset Olivetti tiene **40 personas** distintas (40 clases). Cada neurona entrega el logit de una persona.

### Pregunta 4
¿Qué función cumple `ReLU` en la capa oculta?

**Respuesta:** aporta **no linealidad**, permitiendo que la red combine los píxeles de forma no lineal para distinguir rostros; sin ella la red sería equivalente a un modelo lineal.

---
# 3. Configurar el entrenamiento

Las etiquetas son números enteros entre:

```text
0 y 39
```

Por eso usaremos:

```python
SparseCategoricalCrossentropy
```

Además:

- el modelo produce **logits**,
- usaremos **Adam**,
- mediremos **accuracy**.

Completa `model.compile()`.


In [ ]:
lr_rate = 0.001

# TODO 2 — Configurar loss, optimizer y métrica.  --> RESUELTO
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_rate),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

print("✓ Modelo configurado.")

### Pregunta 5
¿Por qué usamos `from_logits=True` en la función de pérdida?

**Respuesta:** porque la última capa `Dense(40)` no aplica Softmax: entrega logits. `from_logits=True` hace que la pérdida calcule el log-softmax internamente de forma numéricamente estable.

---
# 4. Entrenar el modelo

Usaremos:

```python
epochs = 30
batch_size = 32
```

El dataset es pequeño, por lo que el entrenamiento debería ser rápido.

Keras realizará internamente:

```text
Forward pass
      ↓
Calcular loss
      ↓
Backpropagation
      ↓
Actualizar parámetros con Adam
```

Completa la llamada a `fit()`.


In [ ]:
epochs = 30
batch_size = 32

# TODO 3 — Entrenar el modelo.  --> RESUELTO
history = model.fit(
    X_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test),
    shuffle=True
)

### Pregunta 6
¿Qué representa una **epoch**?

**Respuesta:** una pasada completa por todo el conjunto de entrenamiento (los 320 rostros), tras la cual la red ha actualizado sus pesos con todos los ejemplos una vez.

### Pregunta 7
¿Qué significa `batch_size=32`?

**Respuesta:** que la red procesa 32 imágenes por paso y actualiza los pesos con el gradiente promediado sobre esas 32 imágenes.

---
## 4.1 Visualizar el entrenamiento

Esta parte está resuelta.

El dataset tiene muy pocos ejemplos comparado con el número de parámetros de la red.

Observa cuidadosamente la diferencia entre entrenamiento y validación.


In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    history.history["accuracy"],
    label="Train accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Evolución del entrenamiento")
plt.legend()

plt.show()


### Pregunta 8

Observando la gráfica: ¿el accuracy de entrenamiento sigue aumentando? ¿y el de validación? ¿Hay overfitting?

**Respuesta:** sí hay **overfitting claro**. La accuracy de entrenamiento sube rápido hasta casi 1.0, pero la de validación se queda por debajo y deja de mejorar. Con solo 320 imágenes de entrenamiento y cientos de miles de parámetros, la red memoriza el train antes de generalizar: la brecha train–validation es la evidencia.

---
# 5. Evaluar el modelo

Ahora evaluaremos el modelo sobre imágenes que no fueron utilizadas para actualizar sus parámetros.

Completa `evaluate()`.


In [ ]:
# TODO 4 — Evaluar.  --> RESUELTO
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    batch_size=batch_size,
    verbose=0
)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")
print(f"Accuracy (%)  : {test_accuracy * 100:.2f}%")

### Pregunta 9

Tenemos **40 clases** pero solo **320 imágenes de entrenamiento**. ¿Es mucho o poco para una red con cientos de miles de parámetros?

**Respuesta:** es **muy poco**: apenas 8 imágenes por persona frente a ~260k parámetros. Hay muchísimos más parámetros que ejemplos, lo que favorece la memorización (overfitting). Aun así la tarea es "fácil" porque los rostros Olivetti están bien alineados y controlados, por lo que el accuracy de test suele ser alto (~85–95 %) pese a los pocos datos.

---
# 6. Logits y Softmax

La salida de nuestro modelo tiene forma:

```text
(batch_size, 40)
```

Cada uno de los 40 valores corresponde al **logit** asociado a una persona.

Los logits no son probabilidades.

Para convertirlos en probabilidades usamos:

```python
keras.ops.softmax(...)
```


In [ ]:
index = 0

image = X_test[index]
true_class = y_test[index]

plt.imshow(image, cmap="gray")
plt.title(f"Persona real: {true_class}")
plt.axis("off")
plt.show()

# Añadimos dimensión de batch:
# (64,64) -> (1,64,64)
image_batch = np.expand_dims(image, axis=0)

# Forward pass.
logits = model(
    image_batch,
    training=False
)

print("Shape de logits:", logits.shape)
print("\nPrimeros 10 logits:")
print(
    keras.ops.convert_to_numpy(logits)[0, :10]
)


In [ ]:
# TODO 5 — Convertir logits a probabilidades.  --> RESUELTO
probabilities = keras.ops.softmax(logits)

# TODO 6 — Obtener la clase con mayor probabilidad.  --> RESUELTO
prediction = keras.ops.argmax(probabilities, axis=1)
prediction = int(keras.ops.convert_to_numpy(prediction)[0])

print()
print("Persona real     :", true_class)
print("Persona predicha :", prediction)

In [ ]:
# Revisamos las cinco clases con mayor probabilidad.

probabilities_np = keras.ops.convert_to_numpy(
    probabilities
)[0]

top5 = np.argsort(
    probabilities_np
)[-5:][::-1]

print("Top 5 predicciones:\n")

for class_id in top5:
    print(
        f"Persona {class_id:2d}: "
        f"{probabilities_np[class_id]:.4f}"
    )

print(
    "\nSuma de probabilidades:",
    probabilities_np.sum()
)


### Pregunta 10
¿Por qué la suma de las 40 probabilidades obtenidas con `Softmax` debe ser aproximadamente `1`?

**Respuesta:** porque Softmax produce una distribución de probabilidad sobre las 40 personas: normaliza los logits de modo que cada valor queda en [0,1] y el conjunto suma 1 (salvo redondeo de punto flotante). Representa "qué tan probable es cada persona" y las probabilidades de todas las opciones deben repartir el 100 %.

---
# 7. Visualizar varias predicciones

Esta parte está resuelta.

Observa especialmente los casos en los que el modelo se equivoca.


In [ ]:
n = 12

logits_batch = model(
    X_test[:n],
    training=False
)

predictions = keras.ops.argmax(
    logits_batch,
    axis=1
)

predictions = keras.ops.convert_to_numpy(
    predictions
)

plt.figure(figsize=(12, 8))

for i in range(n):

    plt.subplot(3, 4, i + 1)

    plt.imshow(
        X_test[i],
        cmap="gray"
    )

    real = y_test[i]
    pred = predictions[i]

    plt.title(
        f"Real: {real}\nPred: {pred}",
        fontsize=9
    )

    plt.xticks([])
    plt.yticks([])

plt.tight_layout()
plt.show()


---
# 8. Reflexión final

### 1. ¿Cuál fue el accuracy final del modelo?

**Respuesta:** típicamente alto, **~85–95 %** en test (valor exacto al ejecutar), gracias a que los rostros están bien alineados y centrados.

### 2. ¿Observaste overfitting? ¿Qué evidencia utilizaste?

**Respuesta:** sí. La evidencia es la brecha entre la accuracy de entrenamiento (cercana a 1.0) y la de validación, y el estancamiento o repunte de la `val_loss` mientras la de train sigue bajando.

### 3. ¿Qué ocurre cuando tenemos muchos parámetros pero pocos ejemplos?

**Respuesta:** el modelo tiende a **memorizar** los datos de entrenamiento (incluido su ruido) en lugar de aprender patrones generales, por lo que generaliza mal. Se mitiga con más datos, data augmentation, regularización (L2, Dropout) o reduciendo la capacidad del modelo.

### 4. Después de `Flatten()`, una cara 64×64 se vuelve un vector de 4096 números. ¿Qué información se pierde?

**Respuesta:** la **estructura espacial** del rostro: la posición relativa de ojos, nariz y boca, los bordes y las formas locales. Para la red densa cada píxel es una entrada independiente sin noción de vecindad.

### 5. ¿Una arquitectura diseñada para imágenes aprovecharía mejor la información espacial?

**Respuesta:** sí, una **CNN** usa filtros locales que preservan la vecindad de los píxeles y detectan rasgos (bordes, texturas, partes del rostro), aprovechando la estructura 2D que la red densa ignora.

---
## Para pensar
Una red fully-connected no incorpora la estructura espacial de la imagen. Esto motiva las **Convolutional Neural Networks (CNNs)**.

---
# Resumen

En este ejercicio aplicaste nuevamente la receta básica de entrenamiento de una red neuronal:

```text
Dataset
   ↓
Train / Test split
   ↓
Sequential
   ↓
Flatten
   ↓
Dense + ReLU
   ↓
Dense(40)
   ↓
Cross-Entropy
   ↓
Adam
   ↓
fit()
   ↓
evaluate()
```

Los componentes de Keras son los mismos que usamos en clase.

Lo que cambió fue:

- el dataset,
- el tamaño de las imágenes,
- el número de clases,
- y la dificultad del problema.
